# D-Wave Financial Analysis

This notebook uses `data/processed/dwave_financials_clean.csv` as the single input for financial analysis. All financial statement values are treated as USD millions. Missing values are flagged and left missing.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:,.3f}".format


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "dwave_financials_clean.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/processed/dwave_financials_clean.csv from the current path.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dwave_financials_clean.csv"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
CHARTS_DIR = PROJECT_ROOT / "outputs" / "charts"
TEXT_DIR = PROJECT_ROOT / "outputs" / "text"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input file: {DATA_PATH}")

Project root: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge
Input file: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\data\processed\dwave_financials_clean.csv


## Load Cleaned Financials

In [2]:
df = pd.read_csv(DATA_PATH, na_values=["NA", "", "TO BE FILLED"])

required_columns = [
    "fiscal_year",
    "revenue",
    "gross_profit",
    "r_and_d",
    "sga",
    "operating_income",
    "net_income",
    "cash_and_equivalents",
    "marketable_securities",
    "total_debt",
    "operating_cash_flow",
    "capex",
    "free_cash_flow",
]

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

for column in df.columns:
    if column != "fiscal_year":
        df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.sort_values("fiscal_year").reset_index(drop=True)
print("Cleaned financials loaded. Values are USD millions except share counts.")
display(df)

Cleaned financials loaded. Values are USD millions except share counts.


,fiscal_year,revenue,gross_profit,r_and_d,sga,operating_income,net_income,cash_and_equivalents,marketable_securities,total_debt,operating_cash_flow,capex,free_cash_flow,shares_outstanding
0,2020,5.160,4.245,20.411,15.301,-31.467,-10.000,NaN,NaN,NaN,-29.300,0.789,-30.089,NaN
1,2021,6.279,4.529,25.401,18.076,-38.948,-31.500,9.500,NaN,12.453,-34.800,1.999,-36.799,"2,817,498.000"
2,2022,7.173,4.250,32.101,31.607,-59.458,-51.500,7.100,NaN,9.902,-45.200,0.498,-45.698,"113,335,530.000"
3,2023,8.758,4.622,37.878,47.290,-80.546,-82.700,41.307,NaN,64.249,-60.600,0.630,-61.230,"161,113,744.000"
4,2024,8.827,5.563,35.300,47.486,-77.223,-143.879,177.980,0.000,30.476,-42.643,2.395,-45.038,"266,595,867.000"
5,2025,24.587,20.306,50.734,69.940,-100.368,-355.062,635.347,249.134,35.959,-71.982,4.307,-76.289,"358,741,605.000"


## Missing Data Flags

In [3]:
analysis_columns = [column for column in required_columns if column != "fiscal_year"]
missing_summary = (
    df[["fiscal_year", *analysis_columns]]
    .set_index("fiscal_year")
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)
missing_summary["total_rows"] = len(df)
missing_summary["missing_pct"] = missing_summary["missing_count"] / len(df)
missing_summary = missing_summary[missing_summary["missing_count"] > 0]

if missing_summary.empty:
    print("No missing data found in required analysis fields.")
else:
    warnings.warn("Missing values found. Calculations will remain blank where inputs are unavailable.", stacklevel=2)
    display(missing_summary)

period_missing = df[["fiscal_year", *analysis_columns]].set_index("fiscal_year").isna().sum(axis=1)
period_missing = period_missing[period_missing > 0].rename("missing_fields")
if not period_missing.empty:
    display(period_missing.to_frame())

C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: UserWarning: Missing values found. Calculations will remain blank where inputs are unavailable.
  exec(code_obj, self.user_global_ns, self.user_ns)


,missing_count,total_rows,missing_pct
cash_and_equivalents,1,6,0.167
marketable_securities,4,6,0.667
total_debt,1,6,0.167


,missing_fields
fiscal_year,
2020,3
2021,1
2022,1
2023,1


## Calculate Metrics

In [4]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator / denominator.replace({0: np.nan})


metrics = df.copy()
metrics["revenue_growth"] = metrics["revenue"].pct_change(fill_method=None)
metrics["gross_margin"] = safe_divide(metrics["gross_profit"], metrics["revenue"])
metrics["r_and_d_pct_revenue"] = safe_divide(metrics["r_and_d"], metrics["revenue"])
metrics["sga_pct_revenue"] = safe_divide(metrics["sga"], metrics["revenue"])
metrics["operating_margin"] = safe_divide(metrics["operating_income"], metrics["revenue"])
metrics["net_margin"] = safe_divide(metrics["net_income"], metrics["revenue"])
metrics["free_cash_flow_margin"] = safe_divide(metrics["free_cash_flow"], metrics["revenue"])
metrics["cash_burn"] = np.where(
    metrics["free_cash_flow"].isna(),
    np.nan,
    np.where(metrics["free_cash_flow"] < 0, -metrics["free_cash_flow"], 0),
)
metrics["cash_runway_years"] = np.where(
    metrics["cash_burn"] > 0,
    metrics["cash_and_equivalents"] / metrics["cash_burn"],
    np.nan,
)
metrics["net_cash"] = (
    metrics["cash_and_equivalents"]
    + metrics["marketable_securities"]
    - metrics["total_debt"]
).round(3)

calculated_columns = [
    "revenue_growth",
    "gross_margin",
    "r_and_d_pct_revenue",
    "sga_pct_revenue",
    "operating_margin",
    "net_margin",
    "free_cash_flow_margin",
    "cash_burn",
    "cash_runway_years",
    "net_cash",
]

for column in calculated_columns:
    if metrics[column].isna().all():
        warnings.warn(f"{column} could not be calculated for any fiscal year.", stacklevel=2)

display(metrics[["fiscal_year", *calculated_columns]])

,fiscal_year,revenue_growth,gross_margin,r_and_d_pct_revenue,sga_pct_revenue,operating_margin,net_margin,free_cash_flow_margin,cash_burn,cash_runway_years,net_cash
0,2020,NaN,0.823,3.956,2.965,-6.098,-1.938,-5.831,30.089,NaN,NaN
1,2021,0.217,0.721,4.045,2.879,-6.203,-5.017,-5.861,36.799,0.258,NaN
2,2022,0.142,0.592,4.475,4.406,-8.289,-7.180,-6.371,45.698,0.155,NaN
3,2023,0.221,0.528,4.325,5.400,-9.197,-9.443,-6.991,61.230,0.675,NaN
4,2024,0.008,0.630,3.999,5.380,-8.748,-16.300,-5.102,45.038,3.952,147.504
5,2025,1.785,0.826,2.063,2.845,-4.082,-14.441,-3.103,76.289,8.328,848.522


## Export Summary Tables

In [5]:
income_statement_analysis = metrics[[
    "fiscal_year",
    "revenue",
    "revenue_growth",
    "gross_profit",
    "r_and_d",
    "sga",
    "operating_income",
    "net_income",
]].copy()

margin_analysis = metrics[[
    "fiscal_year",
    "gross_margin",
    "r_and_d_pct_revenue",
    "sga_pct_revenue",
    "operating_margin",
    "net_margin",
    "free_cash_flow_margin",
]].copy()

cash_flow_liquidity_analysis = metrics[[
    "fiscal_year",
    "operating_cash_flow",
    "capex",
    "free_cash_flow",
    "free_cash_flow_margin",
    "cash_burn",
    "cash_runway_years",
]].copy()

balance_sheet_net_cash_analysis = metrics[[
    "fiscal_year",
    "cash_and_equivalents",
    "marketable_securities",
    "total_debt",
    "net_cash",
    "shares_outstanding",
]].copy()

tables = {
    "income_statement_analysis": income_statement_analysis,
    "margin_analysis": margin_analysis,
    "cash_flow_liquidity_analysis": cash_flow_liquidity_analysis,
    "balance_sheet_net_cash_analysis": balance_sheet_net_cash_analysis,
    "financial_metrics_full": metrics,
}

for name, table in tables.items():
    output_path = TABLES_DIR / f"{name}.csv"
    table.to_csv(output_path, index=False, na_rep="NA")
    print(f"Saved table: {output_path}")

display(income_statement_analysis)
display(margin_analysis)
display(cash_flow_liquidity_analysis)
display(balance_sheet_net_cash_analysis)

Saved table: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\tables\income_statement_analysis.csv
Saved table: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\tables\margin_analysis.csv
Saved table: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\tables\cash_flow_liquidity_analysis.csv
Saved table: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\tables\balance_sheet_net_cash_analysis.csv
Saved table: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\tables\financial_metrics_full.csv


,fiscal_year,revenue,revenue_growth,gross_profit,r_and_d,sga,operating_income,net_income
0,2020,5.160,NaN,4.245,20.411,15.301,-31.467,-10.000
1,2021,6.279,0.217,4.529,25.401,18.076,-38.948,-31.500
2,2022,7.173,0.142,4.250,32.101,31.607,-59.458,-51.500
3,2023,8.758,0.221,4.622,37.878,47.290,-80.546,-82.700
4,2024,8.827,0.008,5.563,35.300,47.486,-77.223,-143.879
5,2025,24.587,1.785,20.306,50.734,69.940,-100.368,-355.062


,fiscal_year,gross_margin,r_and_d_pct_revenue,sga_pct_revenue,operating_margin,net_margin,free_cash_flow_margin
0,2020,0.823,3.956,2.965,-6.098,-1.938,-5.831
1,2021,0.721,4.045,2.879,-6.203,-5.017,-5.861
2,2022,0.592,4.475,4.406,-8.289,-7.180,-6.371
3,2023,0.528,4.325,5.400,-9.197,-9.443,-6.991
4,2024,0.630,3.999,5.380,-8.748,-16.300,-5.102
5,2025,0.826,2.063,2.845,-4.082,-14.441,-3.103


,fiscal_year,operating_cash_flow,capex,free_cash_flow,free_cash_flow_margin,cash_burn,cash_runway_years
0,2020,-29.300,0.789,-30.089,-5.831,30.089,NaN
1,2021,-34.800,1.999,-36.799,-5.861,36.799,0.258
2,2022,-45.200,0.498,-45.698,-6.371,45.698,0.155
3,2023,-60.600,0.630,-61.230,-6.991,61.230,0.675
4,2024,-42.643,2.395,-45.038,-5.102,45.038,3.952
5,2025,-71.982,4.307,-76.289,-3.103,76.289,8.328


,fiscal_year,cash_and_equivalents,marketable_securities,total_debt,net_cash,shares_outstanding
0,2020,NaN,NaN,NaN,NaN,NaN
1,2021,9.500,NaN,12.453,NaN,"2,817,498.000"
2,2022,7.100,NaN,9.902,NaN,"113,335,530.000"
3,2023,41.307,NaN,64.249,NaN,"161,113,744.000"
4,2024,177.980,0.000,30.476,147.504,"266,595,867.000"
5,2025,635.347,249.134,35.959,848.522,"358,741,605.000"


## Export Charts

In [6]:
plt.style.use("default")


def save_multi_line_chart(data: pd.DataFrame, columns: list[str], title: str, ylabel: str, filename: str, as_percent: bool = False) -> None:
    available_columns = [column for column in columns if column in data.columns and data[column].notna().any()]
    if not available_columns:
        warnings.warn(f"Skipping {filename}: no available values for {columns}.", stacklevel=2)
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    for column in available_columns:
        chart_data = data[["fiscal_year", column]].dropna()
        y_values = chart_data[column] * 100 if as_percent else chart_data[column]
        ax.plot(chart_data["fiscal_year"].astype(str), y_values, marker="o", linewidth=2, label=column)

    ax.set_title(title)
    ax.set_xlabel("Fiscal year")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    output_path = CHARTS_DIR / filename
    fig.savefig(output_path, dpi=150)
    plt.close(fig)
    print(f"Saved chart: {output_path}")


def save_revenue_growth_chart(data: pd.DataFrame) -> None:
    if data["revenue"].notna().sum() == 0:
        warnings.warn("Skipping revenue_and_growth.png: revenue is missing for all years.", stacklevel=2)
        return

    fig, ax1 = plt.subplots(figsize=(9, 5))
    revenue_data = data[["fiscal_year", "revenue"]].dropna()
    ax1.bar(revenue_data["fiscal_year"].astype(str), revenue_data["revenue"], alpha=0.75, label="revenue")
    ax1.set_xlabel("Fiscal year")
    ax1.set_ylabel("Revenue (USD millions)")
    ax1.grid(True, axis="y", alpha=0.3)

    growth_data = data[["fiscal_year", "revenue_growth"]].dropna()
    if not growth_data.empty:
        ax2 = ax1.twinx()
        ax2.plot(growth_data["fiscal_year"].astype(str), growth_data["revenue_growth"] * 100, color="tab:red", marker="o", linewidth=2, label="revenue_growth")
        ax2.set_ylabel("Revenue growth (%)")
    else:
        warnings.warn("Revenue growth is missing for all years except the first available period.", stacklevel=2)

    ax1.set_title("Revenue And Revenue Growth")
    fig.tight_layout()
    output_path = CHARTS_DIR / "revenue_and_revenue_growth.png"
    fig.savefig(output_path, dpi=150)
    plt.close(fig)
    print(f"Saved chart: {output_path}")


save_revenue_growth_chart(metrics)
save_multi_line_chart(metrics, ["gross_margin"], "Gross Margin", "Gross margin (%)", "gross_margin.png", as_percent=True)
save_multi_line_chart(metrics, ["r_and_d_pct_revenue", "sga_pct_revenue"], "R&D And SG&A As % Of Revenue", "% of revenue", "rd_and_sga_pct_revenue.png", as_percent=True)
save_multi_line_chart(metrics, ["operating_income", "net_income"], "Operating Income And Net Income", "USD millions", "operating_income_and_net_income.png")
save_multi_line_chart(metrics, ["operating_cash_flow", "free_cash_flow"], "Operating Cash Flow And Free Cash Flow", "USD millions", "operating_cash_flow_and_free_cash_flow.png")
save_multi_line_chart(metrics, ["cash_and_equivalents", "marketable_securities", "total_debt", "net_cash"], "Cash, Securities, Debt, And Net Cash", "USD millions", "cash_securities_debt_net_cash.png")

Saved chart: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\charts\revenue_and_revenue_growth.png
Saved chart: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\charts\gross_margin.png


Saved chart: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\charts\rd_and_sga_pct_revenue.png


Saved chart: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\charts\operating_income_and_net_income.png
Saved chart: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\charts\operating_cash_flow_and_free_cash_flow.png


Saved chart: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\charts\cash_securities_debt_net_cash.png


## Export Written Summary

In [7]:
def fmt_amount(value: float) -> str:
    if pd.isna(value):
        return "NA"
    sign = "-" if value < 0 else ""
    return f"{sign}${abs(value):,.1f} million"


def fmt_pct(value: float) -> str:
    if pd.isna(value):
        return "NA"
    return f"{value * 100:,.1f}%"


def fmt_year(value: float) -> str:
    if pd.isna(value):
        return "NA"
    return f"{value:,.1f} years"


latest = metrics.dropna(subset=["fiscal_year"]).sort_values("fiscal_year").iloc[-1]
first = metrics.dropna(subset=["fiscal_year"]).sort_values("fiscal_year").iloc[0]
missing_fields = missing_summary.index.tolist() if not missing_summary.empty else []

summary_lines = [
    "# D-Wave Financial Analysis Summary",
    "",
    "All financial statement values are in USD millions, except share count. No missing values were filled or invented.",
    "",
    f"The dataset covers fiscal {int(first['fiscal_year'])} through fiscal {int(latest['fiscal_year'])}.",
    f"Latest reported revenue was {fmt_amount(latest['revenue'])}, with revenue growth of {fmt_pct(latest['revenue_growth'])}.",
    f"Latest gross margin was {fmt_pct(latest['gross_margin'])}.",
    f"Latest R&D and SG&A were {fmt_pct(latest['r_and_d_pct_revenue'])} and {fmt_pct(latest['sga_pct_revenue'])} of revenue, respectively.",
    f"Latest operating margin and net margin were {fmt_pct(latest['operating_margin'])} and {fmt_pct(latest['net_margin'])}, respectively.",
    f"Latest free cash flow was {fmt_amount(latest['free_cash_flow'])}, implying cash burn of {fmt_amount(latest['cash_burn'])}.",
    f"Latest cash runway, based on cash and equivalents divided by cash burn, was {fmt_year(latest['cash_runway_years'])}.",
    f"Latest net cash was {fmt_amount(latest['net_cash'])}, calculated as cash plus marketable securities minus total debt when all inputs are available.",
    "",
    "## Missing Data Flags",
]

if missing_fields:
    summary_lines.extend(f"- `{field}` has missing values in at least one fiscal year." for field in missing_fields)
else:
    summary_lines.append("- No missing values in required analysis fields.")

summary_lines.extend([
    "",
    "## Exported Tables",
    "- `outputs/tables/income_statement_analysis.csv`",
    "- `outputs/tables/margin_analysis.csv`",
    "- `outputs/tables/cash_flow_liquidity_analysis.csv`",
    "- `outputs/tables/balance_sheet_net_cash_analysis.csv`",
    "- `outputs/tables/financial_metrics_full.csv`",
    "",
    "## Exported Charts",
    "- `outputs/charts/revenue_and_revenue_growth.png`",
    "- `outputs/charts/gross_margin.png`",
    "- `outputs/charts/rd_and_sga_pct_revenue.png`",
    "- `outputs/charts/operating_income_and_net_income.png`",
    "- `outputs/charts/operating_cash_flow_and_free_cash_flow.png`",
    "- `outputs/charts/cash_securities_debt_net_cash.png`",
])

summary_path = TEXT_DIR / "financial_analysis_summary.md"
summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
print(f"Saved written summary: {summary_path}")
print("\n".join(summary_lines))

Saved written summary: C:\Users\ferre\OneDrive - ISE Business School\Belavista\Y3S1 Finanças I\dwave_cfa_challenge\outputs\text\financial_analysis_summary.md
# D-Wave Financial Analysis Summary

All financial statement values are in USD millions, except share count. No missing values were filled or invented.

The dataset covers fiscal 2020 through fiscal 2025.
Latest reported revenue was $24.6 million, with revenue growth of 178.5%.
Latest gross margin was 82.6%.
Latest R&D and SG&A were 206.3% and 284.5% of revenue, respectively.
Latest operating margin and net margin were -408.2% and -1,444.1%, respectively.
Latest free cash flow was -$76.3 million, implying cash burn of $76.3 million.
Latest cash runway, based on cash and equivalents divided by cash burn, was 8.3 years.
Latest net cash was $848.5 million, calculated as cash plus marketable securities minus total debt when all inputs are available.

## Missing Data Flags
- `cash_and_equivalents` has missing values in at least one fis